# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** [Enter Name]
**Student ID:** [Enter ID]

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [21]:
!pip install -q openai python-dotenv pandas matplotlib

In [22]:
# API-key setup — DO NOT hard-code your key in this cell.

import os
import pandas as pd
import matplotlib.pyplot as plt
import json

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY").strip()

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [23]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content



In [24]:

# TODO: Call it once with a simple question and print the answer.
answer = ask_llm("What is the capital of Ghana")
print(answer)


The capital of Ghana is Accra.


In [25]:
# TODO: Print response.usage as well — how many tokens did your call consume?

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "What is the capital of Ghana?"}
    ],
    temperature=0.7,
    max_tokens=500,
)

print(response.usage)

CompletionUsage(completion_tokens=9, prompt_tokens=42, total_tokens=51, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.062105836, prompt_time=0.001958057, completion_time=0.011811491, total_time=0.013769548)


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:** 1. The system role gives the LLM instructions about how it should behave, while the user role contains the specific request or task. For example, the system can tell the user to act as a neutral and  helpful assistant, while the user could ask it to summarize a loan application.

2. A token is a small unit of text an LLM processes. A token can be a whole word or parts of a word. API providers charge based on tokens because longer inputs and outputs require more processing than shorter ones.

### Part 1.2 — Temperature: the randomness dial

In [ ]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
question = "Suggest a name for a savings product for market traders in Accra."

# TODO: Print all 10 answers, grouped by temperature.
print("Temperature = 0.0")
for i in range(5):
    answer = ask_llm(question, temperature=0.0)
    print(f"{i+1}. {answer}")

print("\nTemperature = 1.2")
for i in range(5):
    answer = ask_llm(question, temperature=1.2)
    print(f"{i+1}. {answer}")


Temperature = 0.0
1. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", so this name incorporates local language and culture.
4. **Market Mobi**: This name is short and catchy, and "Mobi" implies mobility and flexibility, which could appeal to market traders.
5. **Sika Kurom**: "Sika" means "money" in Ghanaian, and "Kurom" means "box" or "container", so this name suggests a safe and secure place to store savings.
6. **Traders' Fund**: This name is straightforward and emphasizes the idea of a collective fund for market traders.
7. **Adanfo Save**: "Adanfo" is a Ghanaian word for "friends" or "partners", so this name suggests a sense of community and cooperation.

Choose the one that r

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:** At temperature 0.0, the resposes were more consistent and similar across the five runs. At temperature 1.2, the responses were more varied and creative, with different suggestions being generated. For the loan decision-support system, a low temperature, such as 0.0, is more appropriate because the system needs to produce factual and consistent summaries and structured information rather than creative responses.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [ ]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [ ]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.

SUMMARY_PROMPT_V1 = "Summarize this:"
print("L002:")
print(ask_llm(SUMMARY_PROMPT_V1 + "\n\n" + LETTERS["L002"], temperature=0))

print("\nL006:")
print(ask_llm(SUMMARY_PROMPT_V1 + "\n\n" + LETTERS["L006"], temperature=0))

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

SUMMARY_SYSTEM_V2 = """You are an assistant to a microfinance loan officer.
Summarize loan applications in a factual and neutral way.
Do not invent or assume any information that is not stated in the application.
Keep the summary to 3-4 sentences."""

SUMMARY_PROMPT_V2 = "Summarize this loan application:\n\n"

print("L002:")
print(ask_llm(
    SUMMARY_PROMPT_V2 + LETTERS["L002"],
    system_prompt=SUMMARY_SYSTEM_V2,
    temperature=0
))

print("\nL006:")
print(ask_llm(
    SUMMARY_PROMPT_V2 + LETTERS["L006"],
    system_prompt=SUMMARY_SYSTEM_V2,
    temperature=0
))
# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:** 1. V1 produced a reasonable summary, but it did not consistently include all of the important information from the applications. For example, the V1 summary for L002 did not mention that Kwame had no collateral, while V2 explicitly included this information. or L006, V2 clearly stated that Kofi had not yet started any of the proposed businesses and that he had no collateral.

2. "No invented details" is essential because loan officers may use the summary when assessing an applicant, so incorrect information could lead to an unfair or financially harmful decision. An LLM may sometimes generate information that is not supported by the input, which is known as hallucination. The instruction helps reduce the risk of the model presenting unsupported information as if it were true.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [ ]:
import json
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0


EXTRACT_PROMPT = """
Extract information from the loan application below.

Return ONLY a valid JSON object with EXACTLY these six keys:

{
  "applicant_name": "string",
  "amount_ghs": "number or null",
  "purpose": "string",
  "monthly_profit_ghs": "number or null",
  "has_collateral_or_guarantor": "boolean",
  "repayment_months": "number or null"
}

Rules:
- Use only information explicitly stated in the letter.
- If a field is not stated, use null.
- Do not guess or infer missing information.
- amount_ghs, monthly_profit_ghs, and repayment_months must be numbers.
- has_collateral_or_guarantor must be true or false.
- Return ONLY the JSON object.

Example:

Loan application:
"My name is Ama Mensah. I run a small bakery and request GHS 6,000
to buy an oven. My monthly profit is GHS 700. My brother will
guarantee the loan. I will repay it over 10 months."

JSON:
{
  "applicant_name": "Ama Mensah",
  "amount_ghs": 6000,
  "purpose": "buy an oven",
  "monthly_profit_ghs": 700,
  "has_collateral_or_guarantor": true,
  "repayment_months": 10
}

Loan application:
{letter_text}
"""



# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).
def extract_fields(letter_text):
    prompt = EXTRACT_PROMPT.replace("{letter_text}",letter_text)

    response = ask_llm(
        prompt,
        temperature=0
    )

    try:
        response = response.strip()

        if response.startswith("```json"):
            response = response[7:]

        if response.startswith("```"):
            response = response[3:]

        if response.endswith("```"):
            response = response[:-3]

        response = response.strip()

        return json.loads(response)

    except json.JSONDecodeError:
        print("Warning: Could not parse the LLM response as JSON.")
        print("Response:", response)
        return None

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.
results = []

for letter_id, letter_text in LETTERS.items():
    result = extract_fields(letter_text)

    if result is not None:
        result["letter_id"] = letter_id
        results.append(result)

df = pd.DataFrame(results)

display(df)

**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:**
1. The few-shot example should not come from the six letters because those letters are the data being processed and evaluated. If one of the actual letters were used as the example, the model could be given information about the expected answer, which would make the evaluation less fair.

2. "Use null, do not guess" is important because some information is not provided in a loan application. The model should distinguish between information that is actually stated and information that is missing. Guessing could create false information that might influence a loan officer's decision.

3. Temperature=0 is appropriate for structured extraction because we want the model to produce consistent results rather than creative or highly varied responses. For example, the same loan application should produce the same applicant name, loan amount, and repayment period each time it is processed.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [ ]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.


BRIEF_PROMPT = """
You are an assistant supporting a microfinance loan officer.

Using the loan application and extracted information below, prepare
a decision-support brief.

Include exactly these sections:

1. Strengths
- List strengths supported by the application.

2. Risks / Red Flags
- List risks or concerns supported by the application.

3. Missing Information
- Identify information or documents the loan officer should request.

4. Suggested Next Step
- Suggest an appropriate next step such as requesting documents,
  inviting the applicant for an interview, or flagging the application
  for senior review.

Important:
- Base your response only on information provided.
- Do not invent facts.
- Do not make assumptions about the applicant.
- The system supports the loan officer but does not make the final decision.
- Do NOT say "approve" or "reject".
- Final loan decisions must be made by a human.

Loan application:
{letter_text}

Extracted information:
{extracted_json}
"""

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.
def generate_brief(letter_text, extracted_json):
    prompt = BRIEF_PROMPT.replace("{letter_text}", letter_text)
    prompt = prompt.replace("{extracted_json}", json.dumps(extracted_json))

    response = ask_llm(
        prompt,
        temperature=0
    )

    return response



In [ ]:
briefs = {}

for letter_id in LETTERS:
    extracted = extract_fields(LETTERS[letter_id])
    briefs[letter_id] = generate_brief(LETTERS[letter_id], extracted)

print("L001: ")
print(briefs["L001"])

print("\nL002:")
print(briefs["L002"])

print("\nL006: ")
print(briefs["L006"])

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:**
1. The brief for L003 identified several appropriate strengths, including the registered business, three apprentices, monthly profit of GHS 2,800, the GHS 5,000 fixed deposit that can be pledged, and the available sales records. It also identified the need to review financial records and supporting information. For L006, the system correctly identified major risks, including the fact that Kofi has not started any of the proposed businesses, has no collateral, has no stated monthly profit, and plans to repay after the businesses become successful. Overall, the system identified the main strengths and red flags in both applications based on the information provided.

2. We forbid the model from saying "approve" or "reject" because the system is intended to support the loan officer rather than replace human judgment. Practically, the LLM may miss important information or make an incorrect assessment, so a human should review the application before making the final decision. Ethically, fully automated loan decisions could unfairly harm applicants, especially if the model produces biased or inaccurate recommendations. Keeping a human involved provides an opportunity to review the evidence and challenge an incorrect recommendation.

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** a9a932c


---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [ ]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).
fields = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]

rows = []

for field in fields:
    row = {"field": field}
    correct = 0

    for letter_id in GOLD:
        predicted = df.loc[df["letter_id"] == letter_id, field].iloc[0]
        expected = GOLD[letter_id][field]

        if field == "applicant_name":
            match = str(predicted).lower() == str(expected).lower()
        else:
            match = predicted == expected

        row[letter_id] = match

        if match:
            correct += 1

    row["accuracy"] = correct / len(GOLD)
    rows.append(row)

accuracy_df = pd.DataFrame(rows)

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

display(accuracy_df)

### Part 4.2 — Reliability: is the system consistent?

In [ ]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.


def extract_fields(letter_text, temperature=0):
    prompt = EXTRACT_PROMPT.replace("{letter_text}", letter_text)

    response = ask_llm(
        prompt,
        temperature=temperature
    )

    try:
        response = response.strip()

        if response.startswith("```json"):
            response = response[7:]

        if response.startswith("```"):
            response = response[3:]

        if response.endswith("```"):
            response = response[:-3]

        response = response.strip()

        return json.loads(response)

    except json.JSONDecodeError:
        print("Warning: Could not parse the LLM response as JSON.")
        print("Response:", response)
        return None


results_0 = []
results_1 = []

for i in range(5):
    results_0.append(extract_fields(LETTERS["L004"], temperature=0))

for i in range(5):
    results_1.append(extract_fields(LETTERS["L004"], temperature=1.0))

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

def reliability_stats(results):
    valid_results = [r for r in results if r is not None]

    if valid_results:
        unique_results = set(
            json.dumps(r, sort_keys=True)
            for r in valid_results
        )
    else:
        unique_results = set()

    return len(valid_results), len(unique_results)


valid_0, unique_0 = reliability_stats(results_0)
valid_1, unique_1 = reliability_stats(results_1)

print("Temperature 0.0")
print("Valid JSON:", valid_0, "/ 5")
print("Unique results:", unique_0)

print("\nTemperature 1.0")
print("Valid JSON:", valid_1, "/ 5")
print("Unique results:", unique_1)


### Part 4.3 — Hallucination probing

In [ ]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

test1_prompt = """
What is the applicant's credit score?

If the credit score is not stated in the loan application,
say that it is not provided. Do not guess.

Loan application:
""" + LETTERS["L002"]

test1_output = ask_llm(
    test1_prompt,
    system_prompt=SUMMARY_SYSTEM_V2,
    temperature=0
)

print("Test!:")
print(test1_output)

weather_text = """
Today's weather in Accra will be partly cloudy with temperatures
between 24 and 30 degrees Celsius. There may be occasional rainfall
in the afternoon. Winds will be moderate from the southwest.
"""

test2_output = extract_fields(weather_text, temperature=0)

print("Test 2: ")
print(test2_output)
# TODO: Record the outputs verbatim below and label each PASS or FAIL.
'''
Test!: PASS
Kwame Boateng is applying for a loan of GHS 25,000 to repair his trotro engine and settle personal debts. He is a commercial driver in Kumasi and expects his business to improve after the festive season. The applicant does not have collateral to offer at the moment. The credit score of the applicant is not provided in the loan application.

Test 2: PASS
{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}
'''

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:**
1. The extraction system achieved 100% accuracy for applicant_name, amount_ghs, has_collateral_or_guarantor, and repayment_months. It achieved 66.7% accuracy for monthly_profit_ghs and 0% accuracy for purpose. The purpose field was the hardest for the model because the wording used by the model did not match the gold-standard wording exactly, even when it described the same general purpose.

2. In the reliability experiment, all five runs at temperature 0.0 produced valid JSON and all five results were identical, giving 1 unique result. At temperature 1.0, all five runs also produced valid JSON and all five results were identical, again giving 1 unique result. This shows that the extractor was highly consistent for L004 in this experiment. However, the experiment only used one application and five runs at each temperature, so more testing would be necessary before assuming that the same consistency would occur across a larger and more varied dataset. For a production system, consistency and valid structured output are important because downstream software depends on predictable data.

3. The system did not hallucinate in either of the two adversarial tests. In the first test, the model was asked for the applicant's credit score even though no credit score was included in L002. The model correctly stated that the credit score was not provided. This was a PASS. In the second test, an irrelevant weather report was provided to the extractor instead of a loan application. The extractor returned None for all six requested fields rather than inventing an applicant or loan information. This was also a PASS. These results suggest that the instructions to use null when information is not stated and not to guess helped reduce hallucination.

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:**
1. If the bank fully automated loan decisions using this system, applicants could be unfairly harmed by incorrect or biased outputs. For example, an applicant who writes poorly in English but operates a solid business could have their application misunderstood or assessed less favorably because of the way they communicate rather than the actual strength of their business. This could unfairly affect their access to credit. Keeping a human loan officer involved allows the recommendation to be reviewed and challenged before a final decision is made.

2. Loan applications contain personal and financial information, so sending them to a third-party API in another country creates privacy and data-protection risks. Before deploying the system at a real Ghanaian microfinance institution, I would check how the API provider stores and processes the data, whether submitted information is used for model training, where the data is stored, what security measures are in place, and whether the service complies with applicable Ghanaian data-protection requirements. I would also consider minimizing the personal information sent to the API where possible.

3. First, I would require a human review point before any final loan decision is made. The loan officer should be able to disagree with or override the system's recommendation. Second, I would implement logging and monitoring so that the application's input, extracted information, model output, and final human decision can be reviewed for errors and patterns of unfairness. I would also provide an appeal or human-review process so that applicants have a way to challenge decisions.

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:**
1.

2. I would not trust this system to run unattended. Although the extraction system produced valid and identical results across all five runs at both temperature 0.0 and 1.0, and it passed both hallucination tests, these experiments were limited. The system was tested on only a small number of applications and adversarial cases, so these results are not enough to guarantee reliable behavior in every situation. The hallucination tests were particularly important because they showed that the system could recognize when information was missing rather than inventing it. However, a human loan officer should still make the final decision.

3. Based on the API usage from my test calls, one application requires roughly 196 tokens as a rough estimate. At 1,000 applications per month, this would be approximately 196,000 tokens per month. The actual number would depend on the length of each application, the prompts, and the number of API calls required for summarization, extraction, and decision support. At this scale, the cost and rate limits of the provider would need to be considered when choosing an API, along with factors such as reliability, privacy, and available free-tier limits.

4. For this task, calling an API is more practical than training my own model because the LLM has already been trained on a very large amount of data and can understand and generate natural language without requiring me to collect a large training dataset or spend significant computing resources. I can instead focus on prompt engineering, evaluation, and designing the system around the model. Training my own model could make sense when I have a large specialized dataset, need greater control over the model, have strict privacy requirements, or need a model specifically optimized for a particular task. However, for this lab's loan-application task, using an existing foundation model is much faster and more practical.


---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.